## Application to Italy Air Quality Data

We estimate LatticeKrig parameters from psuedoreplicates of ARX(1) residuals on the NO2 data. 

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import h5py
import time

from latticevision.device import set_device
from latticevision.seed import set_all_random_seeds
from latticevision.img2img import TransUNet
from latticevision.img2img.base import (
	ModelConfig,
	RotaryPosEmbed,
)

In [ ]:
# fix random seed for reproducibility
set_all_random_seeds(777)

device = set_device(machine="remote", gpu_id="cuda:0", verbose=True)  # for remote use
# device = set_device(machine="local", gpu = False, verbose=True)  # for local use

In [ ]:
# loading in the data
file_path = "../data/NO2_resid_df_2023padded.h5"

with h5py.File(file_path, "r") as f:
	print("Components in the file:", list(f.keys()))
	arx1_resid = f["arx1_resid"][:]


print("arx1_resid shape:", arx1_resid.shape)
arx1_resid = np.flip(arx1_resid, axis=1).copy()
arx1_resid = torch.tensor(arx1_resid).float().to(device).unsqueeze(0)
# should have 21 extra days in front and 15 extra days at end due to padding
print("Arx1 residual shape:", arx1_resid.shape)

In [ ]:
plot_ind = 76 + 21

plt.figure(figsize=(4, 4))

# just plot the arx1 resid alone
plt.imshow(
	arx1_resid[0, plot_ind, :, :].detach().cpu().numpy(), aspect="auto", cmap="turbo"
)
plt.colorbar(shrink=0.8)
plt.title("ARX1 Resid")

In [ ]:
# rolling monthly window normalization
def surrounding_norm(
	fields: torch.Tensor, before: int = 15, after: int = 14
) -> torch.Tensor:
	"""
	Normalize each time slice using mean/std computed from surrounding slices including t.
	fields: (1, T, H, W)
	Returns: (1, T, H, W)

	Notes:
	- Uses a target window size of before + 1 + after (default 30).
	- At dataset boundaries, the window is shifted to stay in-range while preserving size when possible.
	- For typical use with T >= 100, every t gets exactly a 30-slice window.
	- Std is clamped to avoid division by zero.
	"""
	assert fields.dim() == 4 and fields.size(0) == 1, "Expected shape (1, T, H, W)"
	_, T, H, W = fields.shape
	out = torch.empty_like(fields)

	for t in range(T):
		# desired window [t-before, t+after], includes t
		start = t - before
		end = t + after
		# Shift the window to keep it within bounds and preserve length when possible
		if start < 0:
			shift = -start
			start = 0
			end = min(T - 1, end + shift)
		if end > T - 1:
			shift = end - (T - 1)
			end = T - 1
			start = max(0, start - shift)

		# Indices including t
		idxs = list(range(start, end + 1))

		# Compute mean/std over the selected time indices (includes t)
		window = fields[:, idxs]  # shape (1, Wlen, H, W)
		m = window.mean(dim=1, keepdim=True)  # (1, 1, H, W)
		s = window.std(dim=1, keepdim=True).clamp_min(1e-8)
		out[:, t : t + 1] = (fields[:, t : t + 1] - m) / s

		# print(t, start, end, shift, idxs)
	return out


def surrounding_normalization(aq_fields: torch.Tensor) -> torch.Tensor:
	with torch.no_grad():
		aq_fields_surround = surrounding_norm(aq_fields, before=15, after=14)
		return aq_fields_surround

In [ ]:
arx1_resid_surround = surrounding_normalization(arx1_resid)
arx1_resid_surround.shape

In [ ]:
# plot a field from each of the 3 options
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.title("ARX1 Resid")
plt.imshow(arx1_resid[0, plot_ind].detach().cpu().numpy(), cmap="turbo")
plt.colorbar(shrink=0.8)

plt.subplot(1, 2, 2)
plt.title("ARX1 Resid (Surround)")
plt.imshow(arx1_resid_surround[0, plot_ind].cpu(), cmap="turbo")
plt.colorbar(shrink=0.8)

## STUN30

In [ ]:
STUN30path = "../results/model_wghts/modelTransUNet_reps30_posRotaryPosEmbed.pth"

config = ModelConfig(
	in_channels=30,
	patch_size_h=2,
	patch_size_w=2,
	pos_embed_cls=RotaryPosEmbed,
)
STUN30 = TransUNet(config)
STUN30.load_state_dict(torch.load(STUN30path))

STUN30 = STUN30.to(device)
STUN30.eval()

In [ ]:
# STUN30 neighborhood parameter extraction
# Computes parameters for each time index using a 30-slice window (15 before, t, 14 after)

def compute_stun30_params(
	fields_1thw: torch.Tensor,
	model: torch.nn.Module,
	before: int = 15,
	after: int = 14,
) -> torch.Tensor:
	if fields_1thw.dim() != 4 or fields_1thw.size(0) != 1:
		raise ValueError("fields_1thw must have shape (1, T, H, W)")

	T = fields_1thw.size(1)
	H = fields_1thw.size(2)
	W = fields_1thw.size(3)
	window_len = before + 1 + after

	if T < window_len:
		raise ValueError(
			f"Need at least {window_len} time steps; got T={T}. Cannot form full window."
		)
	if getattr(model, "in_channels", window_len) != window_len:
		raise ValueError(
			f"Model in_channels ({getattr(model, 'in_channels', 'unknown')}) does not match window length {window_len}."
		)

	device = fields_1thw.device
	outputs = torch.empty((T, 3, H, W), device=device)

	with torch.no_grad():
		for t in range(T):
			start = t - before
			end = t + after

			if start < 0:
				shift = -start
				start = 0
				end = min(T - 1, end + shift)
			if end > T - 1:
				shift = end - (T - 1)
				end = T - 1
				start = max(0, start - shift)

			idxs = list(range(start, end + 1))
			if len(idxs) != window_len:
				# This shouldn't happen given the shifting logic; guard anyway.
				raise RuntimeError(
					f"Window length mismatch at t={t}: expected {window_len}, got {len(idxs)}"
				)

			window = fields_1thw[:, idxs]  # (1, 30, H, W)
			out_t = model(window)  # (1, 3, H, W)
			outputs[t] = out_t[0]
			# print(t, start, end, shift, idxs) # for sanity

	return outputs

In [ ]:
# timing
inference_times = []
for _ in range(30):
	start_time = time.time()
	with torch.no_grad():
		compute_stun30_params(arx1_resid_surround, STUN30)
	end_time = time.time()
	inference_times.append(end_time - start_time)

avg_inference_time = sum(inference_times) / len(inference_times)
print(f"Average time (30 runs): {avg_inference_time} seconds")

In [ ]:
arx1_surround_30rep_output = compute_stun30_params(arx1_resid_surround, STUN30)
print("30-replicate output shape with surround:", arx1_surround_30rep_output.shape)

In [ ]:
arx1_surround_30rep_output = arx1_surround_30rep_output[21:-15, :, :, :]
arx1_surround_30rep_output[:, 1, :, :] = -arx1_surround_30rep_output[:, 1, :, :]

arx1_resid = arx1_resid[0, 21:-15, :, :]
arx1_resid_surround = arx1_resid_surround[0, 21:-15, :, :]

In [ ]:
arx1_surround_30rep_output.shape, arx1_resid.shape, arx1_resid_surround.shape

In [ ]:
def plot_params_with_residual(
	residual_1thw,
	params_tchw,
	plot_idx: int,
	titles=None,
	cmap_resid: str = "turbo",
	cmap_param: str = "viridis",
	figscale=(14, 3.5),
	colorbar: bool = True,
):
	"""
	Plot the residual field and its predicted parameter maps for a given index.

	Arguments
	---------
	residual_1thw: torch.Tensor | np.ndarray
	    Residual tensor shaped (1, T, H, W) or (T, H, W).
	params_tchw: torch.Tensor | np.ndarray
	    Parameters tensor shaped (T, C=3, H, W). Channel 0=kappa2, 1=theta, 2=rho_x.
	plot_idx: int
	    Time index to visualize.
	titles: list[str] | None
	    Custom titles [residual, kappa2, theta, rho_x]. If None, defaults used.
	cmap_resid: str
	    Colormap for residual plot.
	cmap_param: str
	    Colormap for parameter plots.
	figscale: tuple
	    Figure size.
	colorbar: bool
	    Whether to draw colorbars.
	"""

	def to_np(x):
		if isinstance(x, torch.Tensor):
			x = x.detach().cpu().numpy()
		return np.asarray(x)

	res = residual_1thw
	# Accept (1, T, H, W) or (T, H, W)
	if isinstance(res, torch.Tensor):
		if res.dim() == 4 and res.size(0) == 1:
			res_np = to_np(res[0, plot_idx])
		elif res.dim() == 3:
			res_np = to_np(res[plot_idx])
		else:
			raise ValueError("residual_1thw must be (1,T,H,W) or (T,H,W)")
	else:
		res = np.asarray(res)
		if res.ndim == 4 and res.shape[0] == 1:
			res_np = res[0, plot_idx]
		elif res.ndim == 3:
			res_np = res[plot_idx]
		else:
			raise ValueError("residual_1thw must be (1,T,H,W) or (T,H,W)")

	params = params_tchw
	if isinstance(params, torch.Tensor):
		if params.dim() != 4 or params.size(1) != 3:
			raise ValueError("params_tchw must be (T, 3, H, W)")
		p0 = to_np(params[plot_idx, 0])  # kappa2
		p1 = to_np(params[plot_idx, 1])  # theta
		p2 = to_np(params[plot_idx, 2])  # rho_x
	else:
		params = np.asarray(params)
		if params.ndim != 4 or params.shape[1] != 3:
			raise ValueError("params_tchw must be (T, 3, H, W)")
		p0 = params[plot_idx, 0]
		p1 = params[plot_idx, 1]
		p2 = params[plot_idx, 2]

	if titles is None:
		titles = ["Residual", "awght", "theta", "rho"]

	plt.figure(figsize=figscale)

	# Residual
	ax = plt.subplot(1, 4, 1)
	im0 = ax.imshow(res_np, aspect="auto", cmap=cmap_resid)
	ax.set_title(titles[0])
	ax.axis("off")
	if colorbar:
		plt.colorbar(im0, ax=ax, shrink=0.8)

	# kappa2
	ax = plt.subplot(1, 4, 2)
	im1 = ax.imshow(np.exp(p0) + 4, aspect="auto", cmap=cmap_param)
	ax.set_title(titles[1])
	ax.axis("off")
	if colorbar:
		plt.colorbar(im1, ax=ax, shrink=0.8)

	# theta
	ax = plt.subplot(1, 4, 3)
	im2 = ax.imshow(p1, aspect="auto", cmap=cmap_param)
	ax.set_title(titles[2])
	ax.axis("off")
	if colorbar:
		plt.colorbar(im2, ax=ax, shrink=0.8)

	# rho_x
	ax = plt.subplot(1, 4, 4)
	im3 = ax.imshow(p2, aspect="auto", cmap=cmap_param)
	ax.set_title(titles[3])
	ax.axis("off")
	if colorbar:
		plt.colorbar(im3, ax=ax, shrink=0.8)

	plt.tight_layout()

In [ ]:
plot_params_with_residual(
	arx1_resid_surround,
	arx1_surround_30rep_output,
	102,
	titles=["30 Rep ARX1 Resid (Surround)", "awght", "theta", "rho"],
)

plot_params_with_residual(
	arx1_resid_surround,
	arx1_surround_30rep_output,
	76,
	titles=["30 Rep ARX1 Resid (Surround)", "awght", "theta", "rho"],
)

In [ ]:
# # saving the normalized residuals
# output_h5_path_1 = "../data/NO2_resid_df_2023padded_norm.h5"
# with h5py.File(output_h5_path_1, "w") as f:
#     f.create_dataset("arx1_resid_surround", data=arx1_resid_surround.detach().cpu().numpy())

# # saving the outputs from STUN
# output_h5_path_2 = "../results/italy_aq_outputs/STUN_param_df_2023padded.h5"
# with h5py.File(output_h5_path_2, "w") as f:
# 	f.create_dataset(
# 		"arx1_surround_30rep_output", data=arx1_surround_30rep_output.cpu().numpy()
# 	)